# EX_02 — Embeddings con Transformers (ejercicios)

**Notebook de referencia:** `notebook/02_Embeddings_Transformers.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Mean pooling

Con `AutoTokenizer` + `AutoModel`, obtén **last_hidden_state** para una frase y calcula el embedding de frase como media sobre tokens (excluyendo padding).


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

# 1. Definir el texto y cargar un modelo/tokenizer estándar (ej. BERT o DistilBERT)
text = "Transformers build contextual embeddings."
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Tokenizar el texto (incluyendo el return_tensors='pt' para trabajar con PyTorch)
# Añadimos padding por si quieres probar con una lista de varias frases en el futuro.
inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

# 3. Pasar los inputs por el modelo (Forward pass)
with torch.no_grad():
    outputs = model(**inputs)

# 4. Obtener el 'last_hidden_state'
# Dimensiones: [batch_size, sequence_length, hidden_size]
last_hidden_state = outputs.last_hidden_state

# 5. Extraer la máscara de atención
# Dimensiones original: [batch_size, sequence_length]
attention_mask = inputs["attention_mask"]

# --- MEAN POOLING (IGNORANDO PADDING) ---

# A. Expandimos la máscara para que coincida con las dimensiones de last_hidden_state
# Pasamos de [batch_size, sequence_length] a [batch_size, sequence_length, hidden_size]
input_mask_expanded = (
    attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
)

# B. Multiplicamos los embeddings por la máscara para poner a 0 los tokens de padding
sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, dim=1)

# C. Sumamos los elementos de la máscara por frase para saber cuántos tokens reales hay.
# Usamos clamp(min=1e-9) para evitar divisiones por cero si hubiera una frase vacía.
sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# D. Calculamos la media (Mean Pooling)
sentence_embedding = sum_embeddings / sum_mask

# El resultado es el embedding final de tu frase
print("Dimensión del embedding de la frase:", sentence_embedding.shape)
print(sentence_embedding)


## Actividad 2 — `sentence-transformers`

Usa `SentenceTransformer` para embedder dos frases y calcula similitud coseno. Comenta brevemente (en inglés en un comentario) por qué suele ser mejor que mean-pooling manual de BERT base.


In [ ]:
# from sentence_transformers import SentenceTransformer
# import numpy as np

# TODO: encode two sentences, cosine similarity


## Actividad 3 — Paráfrasis

Escribe dos paráfrasis de una misma idea y muestra que sus embeddings (sentence-transformers) tienen **mayor** similitud entre sí que con una frase de tema distinto.


In [ ]:
# TODO: three strings, 2 paraphrase + 1 unrelated, print cosine sims
